# National accounts analysis

## Purpose and scope

This notebook checks the national and global Green Carbon Wealth (GCW), Blue Carbon Wealth (BCW), and Natural Carbon Wealth (NCW) accounts in the current paper-facing exports. It displays the account components, verifies the row-level combination where both accounts are present, and summarizes the current country/territory distribution.

The notebook reads the canonical 2024 country table and current manuscript tables. It does not rebuild underlying forest or marine inputs and does not assign scientific meaning to country rankings.

## 1. Locate and load the current account tables

`Supplementary_country_level_accounts_2024.csv` is the detailed country/territory export. `Table1_global_carbon_wealth.csv` contains the global account totals and uncertainty summaries; `Table2_geographic_economic_profiles.csv` contains the paper's group/region profiles.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'pyproject.toml').is_file():
    ROOT = ROOT.parent
if not (ROOT / 'pyproject.toml').is_file():
    raise FileNotFoundError('Run this notebook from within the repository.')

DATA = ROOT / 'data' / 'derived'
RESULTS = ROOT / 'results'
TABLES = RESULTS / 'tables' / 'manuscript'
VALIDATION = RESULTS / 'validation'
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)
print(f'Repository: {ROOT}')

Repository: C:\Users\aurel\Desktop\M2\Stage ecoseas-csm\CSM\BNC2 paper\repo


## 2. Inspect global accounts and group profiles

Display the compact paper tables first. Check the currency year and annual units in their headers/footnotes before comparing monetary quantities. GCW, BCW and NCW use one valuation framework, but their physical inputs retain their source-specific definitions.

In [2]:
accounts = pd.read_csv(TABLES / 'Supplementary_country_level_accounts_2024.csv')
global_table = pd.read_csv(TABLES / 'Table1_global_carbon_wealth.csv')
profiles = pd.read_csv(TABLES / 'Table2_geographic_economic_profiles.csv')
display(global_table)
display(profiles)

,component,value_mean_trillion_usd2015_yr,value_p05_trillion_usd2015_yr,value_p50_trillion_usd2015_yr,value_p95_trillion_usd2015_yr,share_of_ncw_pct,gcw_records,bcw_records_including_source_aggregate,bcw_country_or_territory_records,units_with_both_accounts,records_with_either_account,aggregation_scope
0,Green carbon wealth,8.261443,3.226506,7.481159,14.999199,78.008686,236,180,179,174,242,Available component total across all GCW units.
1,Blue carbon wealth,2.328971,0.908838,2.113162,4.297896,21.991314,236,180,179,174,242,Available component total; BCW includes the so...
2,Natural carbon wealth,10.590414,4.163492,9.600348,19.204331,100.000000,236,180,179,174,242,Global GCW + global BCW; national NCW remains ...


,group,units_with_gcw,units_with_gcw_and_bcw,units_with_2024_gdp,gcw_trillion_usd2015_yr,median_gcw_gdp_2024_pct,green_dominant_units,mixed_units,blue_dominant_units
0,Developed economies,62,37,50,3.611950,3.988163,22,6,9
1,Developing economies,80,59,67,3.430066,8.068553,29,14,16
2,LDCs,37,21,34,1.054454,45.244035,12,6,3
3,SIDS,57,57,45,0.164973,4.670785,8,12,37
4,Total,236,174,196,8.261443,8.274631,71,38,65


## 3. Check the unit-level account identity

For records with both GCW and BCW, compare `GCW + BCW` with NCW. The maximum absolute residual is a numeric consistency check, not an additional estimate. Records without one component are omitted from this particular identity check rather than treated as zero.

In [3]:
gcw = pd.to_numeric(accounts['gcw_usd2015_yr_mean'], errors='coerce')
bcw = pd.to_numeric(accounts['bcw_usd2015_yr_mean'], errors='coerce')
ncw = pd.to_numeric(accounts['ncw_usd2015_yr_mean'], errors='coerce')
valid = gcw.notna() & bcw.notna() & ncw.notna()
identity = pd.DataFrame({'UNIT_CODE': accounts.loc[valid, 'UNIT_CODE'], 'name': accounts.loc[valid, 'name'], 'gcw_plus_bcw': (gcw[valid] + bcw[valid]), 'ncw': ncw[valid]})
identity['absolute_difference_usd2015_yr'] = (identity['gcw_plus_bcw'] - identity['ncw']).abs()
print('Records checked:', len(identity))
print('Maximum absolute difference (US$2015/year):', identity['absolute_difference_usd2015_yr'].max())
display(identity.sort_values('absolute_difference_usd2015_yr', ascending=False).head(10))

Records checked: 174
Maximum absolute difference (US$2015/year): 80.0


,UNIT_CODE,name,gcw_plus_bcw,ncw,absolute_difference_usd2015_yr
47,CAN,Canada,6.445575e+11,6.445575e+11,80.0
113,IDN,Indonesia,2.671673e+11,2.671673e+11,80.0
50,CHL,Chile,1.819076e+11,1.819076e+11,70.0
87,FRA,France,1.447353e+11,1.447353e+11,50.0
115,IND,India,2.244870e+11,2.244870e+11,40.0
8,ARG,Argentina,1.228415e+11,1.228415e+11,30.0
3,AGO,Angola,1.462802e+11,1.462802e+11,30.0
127,JPN,Japan,1.127325e+11,1.127325e+11,30.0
186,PER,Peru,2.217740e+11,2.217740e+11,30.0
201,RUS,Russian Federation,1.293529e+12,1.293529e+12,30.0


## 4. Inspect national and territory distributions

This table ranks the unit records with non-missing NCW. Review the `unit_class` and `exclude_from_country_sum` fields before aggregating units: the study unit crosswalk includes territories and accounting units that should not automatically be collapsed or added as sovereign countries.

In [4]:
ranked = accounts.loc[ncw.notna(), ['UNIT_CODE', 'name', 'region', 'groups', 'unit_class', 'exclude_from_country_sum', 'gcw_usd2015_yr_mean', 'bcw_usd2015_yr_mean', 'ncw_usd2015_yr_mean']].copy()
display(ranked.sort_values('ncw_usd2015_yr_mean', ascending=False).head(20))
print('Country/territory records:', len(accounts), '| records with NCW:', int(ncw.notna().sum()))

,UNIT_CODE,name,region,groups,unit_class,exclude_from_country_sum,gcw_usd2015_yr_mean,bcw_usd2015_yr_mean,ncw_usd2015_yr_mean
201,RUS,Russian Federation,Europe,Developed economies,FRA country or territory,False,1.195472e+12,9.805708e+10,1.293529e+12
251,USA,United States of America,Americas,Developed economies,FRA country or territory,False,8.194324e+11,1.711785e+11,9.906109e+11
40,BRA,Brazil,Americas,Developing economies,FRA country or territory,False,9.383748e+11,3.019182e+10,9.685667e+11
47,CAN,Canada,Americas,Developed economies,FRA country or territory,False,6.093673e+11,3.519028e+10,6.445575e+11
51,CHN,"China (incl. Taiwan, Hong Kong SAR and Macao SAR)",Asia,Developing economies,FRA country or territory,False,6.101334e+11,2.190134e+10,6.320348e+11
20,AUS,Australia,Oceania,Developed economies,FRA country or territory,False,3.633880e+11,1.404477e+11,5.038357e+11
113,IDN,Indonesia,Asia,Developing economies,FRA country or territory,False,1.839893e+11,8.317793e+10,2.671673e+11
54,COD,Democratic Republic of the Congo,Africa,LDCs,FRA country or territory,False,2.668489e+11,7.155083e+07,2.669204e+11
115,IND,India,Asia,Developing economies,FRA country or territory,False,1.525161e+11,7.197088e+10,2.244870e+11
153,MEX,Mexico,Americas,Developing economies,FRA country or territory,False,1.524336e+11,7.148156e+10,2.239151e+11


Country/territory records: 267 | records with NCW: 174


## 5. Review concentration and supplementary profiles

The concentration and regional-profile tables provide additional distributional summaries. They are displayed as calculated outputs, with no automatic choice of which concentration statistic or grouping belongs in the manuscript.

In [5]:
concentration = pd.read_csv(TABLES / 'Supplementary_account_concentration_2024.csv')
regions = pd.read_csv(TABLES / 'Supplementary_region_profiles_2024.csv')
display(concentration)
display(regions)

,measure,n_units,gini,top_10_share_pct,top_50_share_pct,median_usd2015_yr,aggregation_scope
0,GCW,236,0.865217,64.056074,90.630345,3.328198e+09,Available component values by country/territor...
1,BCW,180,NaN,44.039313,88.043774,2.487513e+09,Available component values by country/territor...
2,NCW,174,0.808192,60.819285,89.557641,1.194002e+10,Country/territory NCW where both accounts are ...


,region,units_with_gcw,units_with_bcw,units_with_both_accounts,gcw_trillion_usd2015_yr,bcw_trillion_usd2015_yr,ncw_trillion_usd2015_yr,ncw_complete_case_trillion_usd2015_yr,units_with_2024_gdp,median_gcw_gdp_2024_pct
0,Africa,58,42,39,1.290881,3.658682e-01,1.656749,1.299677,52,32.676573
1,Americas,27,22,22,3.291339,7.081588e-01,3.999498,3.867848,23,31.394834
2,Asia,48,34,34,1.488393,3.811324e-01,1.869526,1.742122,45,4.441197
3,Caribbean,28,28,28,0.016555,5.926394e-02,0.075819,0.075819,19,2.075410
4,Europe,50,29,29,1.707374,4.107565e-01,2.118131,2.035000,42,4.152050
5,Oceania,25,24,22,0.466901,4.037904e-01,0.870692,0.870515,15,21.732713
6,Unclassified,0,1,0,NaN,9.736643e-07,NaN,NaN,0,NaN
7,All units,236,180,174,8.261443,2.328971e+00,10.590414,9.890982,196,8.274631
